In [42]:
import requests
from bs4 import BeautifulSoup
import pandas as pd
import time

# Headers
headers = {
    "User-Agent": "Mozilla/5.0"
}

# ✅ Correct domains
domains = {
    "Data Analyst": "data-analytics-internship",
    "Web Development": "web-development-internship",
    "Software Development": "software-development-internship",
    "Data Science": "data-science-internship",
    "UI/UX Design": "ui-ux-design-internship"
}

base_url = "https://internshala.com/internships"

all_jobs = []

# 🔁 Loop through domains
for domain_name, domain_slug in domains.items():
    print(f"\n🔍 Scraping {domain_name}...")

    # 🔁 Keep pages small (more stable)
    for page in range(1, 4):

        if page == 1:
            url = f"{base_url}/{domain_slug}/"
        else:
            url = f"{base_url}/{domain_slug}/page-{page}/"

        print(f"➡️ {url}")

        response = requests.get(url, headers=headers)
        soup = BeautifulSoup(response.content, "html.parser")

        jobs = soup.select("div.individual_internship")

        print(f"📊 Jobs found: {len(jobs)}")

        # ❌ DO NOT BREAK → skip instead
        if not jobs:
            print("⚠️ No jobs found, skipping page...")
            continue

        # 🔍 Extract data
        for job in jobs:
            try:
                title = job.select_one("a.job-title-href").text.strip()
                company = job.select_one("p.company-name").text.strip()

                # ✅ LOCATION (robust)
                location_tags = job.select("div.row-1-item.locations a")
                if not location_tags:
                    location_tags = job.select("a.location_link")

                location = ", ".join([loc.text.strip() for loc in location_tags]) if location_tags else "N/A"

                # ✅ STIPEND
                stipend_tag = job.select_one("span.stipend")
                stipend = stipend_tag.text.strip() if stipend_tag else "Not Disclosed"

                # ✅ START DATE & DURATION (robust)
                start_date = "N/A"
                duration = "N/A"

                detail_blocks = job.select("div.internship_other_details_container div.other_detail_item")

                for block in detail_blocks:
                    label_tag = block.select_one("span")
                    value_tag = block.select_one("div")

                    if label_tag and value_tag:
                        label = label_tag.text.strip().lower()
                        value = value_tag.text.strip()

                        if "start" in label:
                            start_date = value

                        elif "duration" in label:
                            duration = value

                # ✅ APPLY BY
                apply_by_tag = job.select_one("div.status-container")
                apply_by = apply_by_tag.text.strip() if apply_by_tag else "N/A"

                # ✅ JOB LINK
                link_tag = job.select_one("a.job-title-href")
                job_link = "https://internshala.com" + link_tag["href"] if link_tag else "N/A"

                all_jobs.append({
                    "domain": domain_name,
                    "job_title": title,
                    "company": company,
                    "location": location,
                    "stipend": stipend,
                    "start_date": start_date,
                    "duration": duration,
                    "apply_by": apply_by,
                    "job_link": job_link
                })

            except Exception:
                continue

       
        time.sleep(2)


df = pd.DataFrame(all_jobs)
df.to_csv("internshala_multi_domain_jobs.csv", index=False)

print("\n✅ Scraping Completed!")
print(f"📁 Total records collected: {len(df)}")


🔍 Scraping Data Analyst...
➡️ https://internshala.com/internships/data-analytics-internship/
📊 Jobs found: 42
➡️ https://internshala.com/internships/data-analytics-internship/page-2/
📊 Jobs found: 40
➡️ https://internshala.com/internships/data-analytics-internship/page-3/
📊 Jobs found: 1

🔍 Scraping Web Development...
➡️ https://internshala.com/internships/web-development-internship/
📊 Jobs found: 50
➡️ https://internshala.com/internships/web-development-internship/page-2/
📊 Jobs found: 40
➡️ https://internshala.com/internships/web-development-internship/page-3/
📊 Jobs found: 40

🔍 Scraping Software Development...
➡️ https://internshala.com/internships/software-development-internship/
📊 Jobs found: 50
➡️ https://internshala.com/internships/software-development-internship/page-2/
📊 Jobs found: 40
➡️ https://internshala.com/internships/software-development-internship/page-3/
📊 Jobs found: 40

🔍 Scraping Data Science...
➡️ https://internshala.com/internships/data-science-internship/
📊 Jo